In [1]:
import sqlite3
import pandas as pd

# 1. Connect to your SQLite database file
# Replace this with the path to your existing kenya_water.db file
db_path = "data/processed/kenya_water.db"
conn = sqlite3.connect(db_path)

# 2. Run a simple query to see your tables
# This query gets the count of tests from your KEWI table
query = "SELECT COUNT(*) AS total_tests FROM kewi_water_tests;"
df = pd.read_sql_query(query, conn)

print(df)

# 3. Close the connection
conn.close()

   total_tests
0          493


In [2]:
import sqlite3
import pandas as pd
import os

os.chdir(r"C:\Users\user\Desktop\Final Project")
conn = sqlite3.connect("data/processed/kenya_water.db")

# ============================================================
# 1. NATIONAL TRENDS (water + sanitation + defecation)
# ============================================================
national_trend = pd.read_sql_query("""
    SELECT 
        year,
        ROUND(MAX(CASE WHEN indicator_code = 'SH.H2O.BASW.ZS' THEN value END), 2) AS water_access_pct,
        ROUND(MAX(CASE WHEN indicator_code = 'SH.STA.BASS.ZS' THEN value END), 2) AS sanitation_access_pct,
        ROUND(MAX(CASE WHEN indicator_code = 'SH.STA.ODFC.ZS' THEN value END), 2) AS open_defecation_pct
    FROM national_indicators
    WHERE indicator_code IN ('SH.H2O.BASW.ZS', 'SH.STA.BASS.ZS', 'SH.STA.ODFC.ZS')
    GROUP BY year
    ORDER BY year
""", conn)
national_trend.to_csv("data/results/01_national_trends.csv", index=False)
print(f"✅ 01_national_trends.csv: {len(national_trend)} rows")
print(national_trend.tail().to_string(index=False))

# ============================================================
# 2. FRESHWATER WITHDRAWAL TREND
# ============================================================
freshwater = pd.read_sql_query("""
    SELECT year, ROUND(value, 2) AS freshwater_withdrawal_pct
    FROM national_indicators
    WHERE indicator_code = 'ER.H2O.FWTL.ZS'
    ORDER BY year
""", conn)
freshwater.to_csv("data/results/02_freshwater_withdrawal.csv", index=False)
print(f"\n✅ 02_freshwater_withdrawal.csv: {len(freshwater)} rows")

# ============================================================
# 3. KEWI — WATER QUALITY BY COUNTY
# ============================================================
kewi_by_county = pd.read_sql_query("""
    SELECT 
        COUNTY,
        COUNT(*) AS total_tests,
        ROUND(AVG(PH), 2) AS avg_ph,
        COUNT(PH) AS ph_tests,
        ROUND(AVG(CONDUCTIVITY), 1) AS avg_conductivity,
        COUNT(CONDUCTIVITY) AS cond_tests,
        ROUND(AVG(Total_Dissolved), 1) AS avg_tds,
        COUNT(Total_Dissolved) AS tds_tests
    FROM kewi_water_tests
    WHERE COUNTY IS NOT NULL
    GROUP BY COUNTY
    ORDER BY avg_conductivity DESC
""", conn)
kewi_by_county.to_csv("data/results/03_kewi_by_county.csv", index=False)
print(f"\n✅ 03_kewi_by_county.csv: {len(kewi_by_county)} rows")
print(kewi_by_county.to_string(index=False))

# ============================================================
# 4. KEWI — WATER QUALITY BY SOURCE TYPE
# ============================================================
kewi_by_source = pd.read_sql_query("""
    SELECT 
        TYPE_OF_ AS source_type,
        COUNT(*) AS total_tests,
        ROUND(AVG(PH), 2) AS avg_ph,
        ROUND(AVG(CONDUCTIVITY), 1) AS avg_conductivity,
        ROUND(AVG(Total_Dissolved), 1) AS avg_tds
    FROM kewi_water_tests
    WHERE TYPE_OF_ IS NOT NULL
    GROUP BY TYPE_OF_
    ORDER BY total_tests DESC
""", conn)
kewi_by_source.to_csv("data/results/04_kewi_by_source.csv", index=False)
print(f"\n✅ 04_kewi_by_source.csv: {len(kewi_by_source)} rows")
print(kewi_by_source.to_string(index=False))

# ============================================================
# 5. ATHI RIVER COVERAGE
# ============================================================
athi = pd.read_sql_query("""
    SELECT 
        Type_of_Coverage,
        Date,
        Total_Pop,
        Coverage_pct,
        Served_Pop,
        Non_served_Pop
    FROM athi_river_coverage
    ORDER BY Type_of_Coverage, Date
""", conn)
athi.to_csv("data/results/05_athi_river_coverage.csv", index=False)
print(f"\n✅ 05_athi_river_coverage.csv: {len(athi)} rows")

# ============================================================
# 6. ATHI RIVER — SERVED VS UNSERVED SUMMARY
# ============================================================
athi_summary = pd.read_sql_query("""
    SELECT 
        Type_of_Coverage,
        SUM(Served_Pop) AS total_served,
        SUM(Non_served_Pop) AS total_unserved,
        ROUND(100.0 * SUM(Served_Pop) / (SUM(Served_Pop) + SUM(Non_served_Pop)), 1) AS served_pct
    FROM athi_river_coverage
    GROUP BY Type_of_Coverage
""", conn)
athi_summary.to_csv("data/results/06_athi_summary.csv", index=False)
print(f"\n✅ 06_athi_summary.csv: {len(athi_summary)} rows")
print(athi_summary.to_string(index=False))

# ============================================================
# 7. KEWI — SAMPLE WATER SOURCE BREAKDOWN
# ============================================================
source_count = pd.read_sql_query("""
    SELECT TYPE_OF_ AS source_type, COUNT(*) AS tests
    FROM kewi_water_tests
    GROUP BY TYPE_OF_
    ORDER BY tests DESC
""", conn)
source_count.to_csv("data/results/07_kewi_source_breakdown.csv", index=False)
print(f"\n✅ 07_kewi_source_breakdown.csv: {len(source_count)} rows")

conn.close()

# Make sure the folder exists
os.makedirs("data/results", exist_ok=True)

print("\n" + "=" * 60)
print("✅ ALL 7 RESULT FILES SAVED TO data/results/")
print("=" * 60)
print("\nOpen these CSVs in Excel to build your dashboard:")
for f in sorted(os.listdir("data/results")):
    print(f"   {f}")

OSError: Cannot save file into a non-existent directory: 'data\results'

In [4]:
import sqlite3
import pandas as pd
import os

os.chdir(r"C:\Users\user\Desktop\Final Project")
os.makedirs("data/results", exist_ok=True)

conn = sqlite3.connect("data/processed/kenya_water.db")

# --- Detect actual column names in kewi_water_tests ---
cols_df = pd.read_sql_query("PRAGMA table_info(kewi_water_tests)", conn)
cols = [c for c in cols_df["name"]]

def find_col(candidates):
    """Return the first matching column name, case-insensitive."""
    for cand in candidates:
        for c in cols:
            if c.lower().strip() == cand.lower():
                return c
    return None

county_col = find_col(["COUNTY", "county"])
source_col = find_col(["TYPE_OF_", "TYPE_OF", "SOURCE", "source_type"])
ph_col     = find_col(["PH", "ph"])
cond_col   = find_col(["CONDUCTIVITY", "conductivity", "COND", "conductivty"])
tds_col    = find_col(["Total_Dissolved", "TDS", "total_dissolved", "total_dissolved_solids"])

print(f"Detected columns:")
print(f"   county       → '{county_col}'")
print(f"   source       → '{source_col}'")
print(f"   pH           → '{ph_col}'")
print(f"   conductivity → '{cond_col}'")
print(f"   TDS          → '{tds_col}'")
print()

if cond_col is None:
    print("⚠️ Could not find conductivity column — check the output above")

# --- Build queries dynamically using the detected column names ---

# 1. NATIONAL TRENDS
national_trend = pd.read_sql_query("""
    SELECT 
        year,
        ROUND(MAX(CASE WHEN indicator_code = 'SH.H2O.BASW.ZS' THEN value END), 2) AS water_access_pct,
        ROUND(MAX(CASE WHEN indicator_code = 'SH.STA.BASS.ZS' THEN value END), 2) AS sanitation_access_pct,
        ROUND(MAX(CASE WHEN indicator_code = 'SH.STA.ODFC.ZS' THEN value END), 2) AS open_defecation_pct
    FROM national_indicators
    WHERE indicator_code IN ('SH.H2O.BASW.ZS', 'SH.STA.BASS.ZS', 'SH.STA.ODFC.ZS')
    GROUP BY year
    ORDER BY year
""", conn)
national_trend.to_csv("data/results/01_national_trends.csv", index=False)
print(f"✅ 01_national_trends.csv: {len(national_trend)} rows")

# 2. FRESHWATER WITHDRAWAL
freshwater = pd.read_sql_query("""
    SELECT year, ROUND(value, 2) AS freshwater_withdrawal_pct
    FROM national_indicators
    WHERE indicator_code = 'ER.H2O.FWTL.ZS'
    ORDER BY year
""", conn)
freshwater.to_csv("data/results/02_freshwater_withdrawal.csv", index=False)
print(f"✅ 02_freshwater_withdrawal.csv: {len(freshwater)} rows")

# 3. KEWI — BY COUNTY (only if we have the columns)
if county_col and cond_col:
    q3 = f"""
        SELECT 
            "{county_col}" AS county,
            COUNT(*) AS total_tests,
            ROUND(AVG("{ph_col}"), 2) AS avg_ph,
            ROUND(AVG("{cond_col}"), 1) AS avg_conductivity
            {', ROUND(AVG("' + tds_col + '"), 1) AS avg_tds' if tds_col else ''}
        FROM kewi_water_tests
        WHERE "{county_col}" IS NOT NULL
        GROUP BY "{county_col}"
        ORDER BY avg_conductivity DESC
    """
    kewi_by_county = pd.read_sql_query(q3, conn)
    kewi_by_county.to_csv("data/results/03_kewi_by_county.csv", index=False)
    print(f"✅ 03_kewi_by_county.csv: {len(kewi_by_county)} rows")
    print(kewi_by_county.to_string(index=False))

# 4. KEWI — BY SOURCE
if source_col and cond_col:
    q4 = f"""
        SELECT 
            "{source_col}" AS source_type,
            COUNT(*) AS total_tests,
            ROUND(AVG("{ph_col}"), 2) AS avg_ph,
            ROUND(AVG("{cond_col}"), 1) AS avg_conductivity
            {', ROUND(AVG("' + tds_col + '"), 1) AS avg_tds' if tds_col else ''}
        FROM kewi_water_tests
        WHERE "{source_col}" IS NOT NULL
        GROUP BY "{source_col}"
        ORDER BY total_tests DESC
    """
    kewi_by_source = pd.read_sql_query(q4, conn)
    kewi_by_source.to_csv("data/results/04_kewi_by_source.csv", index=False)
    print(f"✅ 04_kewi_by_source.csv: {len(kewi_by_source)} rows")
    print(kewi_by_source.to_string(index=False))

# 5. ATHI RIVER
athi = pd.read_sql_query("""
    SELECT * FROM athi_river_coverage
""", conn)
athi.to_csv("data/results/05_athi_river_coverage.csv", index=False)
print(f"✅ 05_athi_river_coverage.csv: {len(athi)} rows")

# 6. ATHI SUMMARY
athi_summary = pd.read_sql_query("""
    SELECT 
        Type_of_Coverage,
        SUM(Served_Pop) AS total_served,
        SUM(Non_served_Pop) AS total_unserved,
        ROUND(100.0 * SUM(Served_Pop) / (SUM(Served_Pop) + SUM(Non_served_Pop)), 1) AS served_pct
    FROM athi_river_coverage
    GROUP BY Type_of_Coverage
""", conn)
athi_summary.to_csv("data/results/06_athi_summary.csv", index=False)
print(f"✅ 06_athi_summary.csv: {len(athi_summary)} rows")

# 7. KEWI — SOURCE BREAKDOWN
if source_col:
    q7 = f"""
        SELECT "{source_col}" AS source_type, COUNT(*) AS tests
        FROM kewi_water_tests
        GROUP BY "{source_col}"
        ORDER BY tests DESC
    """
    source_count = pd.read_sql_query(q7, conn)
    source_count.to_csv("data/results/07_kewi_source_breakdown.csv", index=False)
    print(f"✅ 07_kewi_source_breakdown.csv: {len(source_count)} rows")

conn.close()

print("\n" + "=" * 60)
print("✅ RESULTS SAVED TO data/results/")
print("=" * 60)
for f in sorted(os.listdir("data/results")):
    print(f"   {f}")

Detected columns:
   county       → 'COUNTY'
   source       → 'None'
   pH           → 'PH'
   conductivity → 'None'
   TDS          → 'Total_Dissolved_Solids'

⚠️ Could not find conductivity column — check the output above
✅ 01_national_trends.csv: 25 rows
✅ 02_freshwater_withdrawal.csv: 23 rows
✅ 05_athi_river_coverage.csv: 8 rows
✅ 06_athi_summary.csv: 2 rows

✅ RESULTS SAVED TO data/results/
   01_national_trends.csv
   02_freshwater_withdrawal.csv
   05_athi_river_coverage.csv
   06_athi_summary.csv
